# Lista 2 - MLP, CNN, GAN, GNN
**Aluno:** Vitor Fontenele de Oliveira Linhares

**Mátricula:** 1700778

## Setup

In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import FashionMNIST, MNIST
from torchvision.utils import make_grid
from ucimlrepo import fetch_ucirepo
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Questão 1: MLP - Breast Cancer Wisconsin

### 1.1 Descrição e entendimento do problema

O dataset reúne atributos numéricos extraídos de imagens de núcleos de células, e tentaremos prever o diagnóstico entre benigno (B) e maligno (M). Inicialmente, faremos uma exploração inicial para entender o formato do dataset, observar o comportamento das variáveis e verificar se existiam sinais de desbalanceamento entre as classes target e correlações relevantes (que não sejam na diagonal principal da matriz de correlação).

In [ ]:
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 

target_col = y['Diagnosis']
display(target_col.value_counts())
target_numeric = target_col.map({'B': 0, 'M': 1})

df = pd.concat([X, target_numeric.rename('Diagnosis_Target')], axis=1)
display(df.head())

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

first_feat = X.columns[0]
sns.histplot(data=df, x=first_feat, hue=target_col, kde=True, palette='Set2', ax=axes[0])
axes[0].set_title(f'Distribuição: {first_feat}')
axes[0].set_xlabel(first_feat)

corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.2, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Matriz de Correlação (features)')

plt.tight_layout()
plt.show()

Na exploração inicial, percebemos alguns pontos importantes. A base tem muitas variáveis com comportamento e nome parecido (e.g., `radius1`, `radius2`, `radius3`), o que indica potencial correlação entre algumas features, e a matriz de correlação confirma isso com vários pares fortes fora da diagonal principal. Também olhamos o balanceamento do target, porque em problema médico tabular é possível que venha desbalanceado. Pudemos perceber que existe diferença, mas não é extrema (**357** benignos e **212** malignos). No fim, o problema realmsente parece se caracterizar e se adequar como classificação supervisionada em dados tabulares, com alvo binário (benigno vs. maligno). 

A próxima fase será a preparação dos dados para MLP

### 1.2 Preparação dos Dados
*Pré-processamento e divisão treino/validação/teste.*

Neste momento, faremos o split em treino, validação e teste. Depois, definiremos uma função para reduzir dimensões avaliando a potencial correlação percebida anteriormente e usaremos para treino e aplicaremos as mesmas colunas em validação e teste. Por fim, padronizamos com `StandardScaler` ajustado apenas no treino.

In [ ]:
def fit_corr_feature_selector(X_train_df, threshold=0.95):
    feature_cols = X_train_df.columns.tolist()

    corr_feat = X_train_df.corr().abs()
    upper_feat = corr_feat.where(np.triu(np.ones(corr_feat.shape), k=1).astype(bool))

    high_pairs_095 = (
        upper_feat.stack()
        .reset_index()
        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'corr_abs'})
    )
    high_pairs_095 = high_pairs_095[high_pairs_095['corr_abs'] >= threshold].reset_index(drop=True)

    adj = {f: set() for f in feature_cols}
    for _, row in high_pairs_095.iterrows():
        a = row['feature_1']
        b = row['feature_2']
        adj[a].add(b)
        adj[b].add(a)

    visited = set()
    components = []
    for f in feature_cols:
        if f in visited:
            continue
        stack = [f]
        comp = set()
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            comp.add(node)
            stack.extend(adj[node] - visited)
        components.append(comp)

    keep_features = [sorted(comp)[0] for comp in components]
    return sorted(keep_features), high_pairs_095

print('Função de redução por correlação pronta.')

Resumo: para evitar vazamento de dados, a redução por correlação passou a ser ajustada apenas no conjunto de treino. Depois aplicamos esse mesmo subconjunto de colunas em validação e teste. Em seguida, padronizamos com `StandardScaler`, com `fit_transform` só no treino e `transform` em validação/teste. Assim a avaliação fica metodologicamente mais correta.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, target_numeric, test_size=0.15, random_state=42, stratify=target_numeric
)

val_ratio_adjusted = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio_adjusted, random_state=42, stratify=y_temp
)

print(f'Features antes da redução (treino): {X_train.shape[1]}')

keep_features, high_pairs_095 = fit_corr_feature_selector(X_train, threshold=0.95)

X_train = X_train[keep_features].copy()
X_val = X_val[keep_features].copy()
X_test = X_test[keep_features].copy()

print(f'Features após redução (fit no treino): {len(keep_features)}')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Shapes -> treino: {X_train.shape}, validação: {X_val.shape}, teste: {X_test.shape}')

Resumo: primeiro separamos treino, validação e teste com estratificação para manter a proporção das classes parecida nos três conjuntos. Depois padronizamos as variáveis com `StandardScaler`. O `fit_transform` fica só no treino porque é nele que o scaler aprende média e desvio padrão; em validação e teste usamos apenas `transform`, reaplicando essa mesma escala. Fazemos assim para evitar vazamento de dados e manter a avaliação mais confiável.

### 1.3 Configuração + Treinamento + Validação

Nesta etapa comparamos duas configurações de MLP. Em cada época calculamos métricas de treino e de validação, e escolhemos o melhor estado do modelo com base na perda de validação. Separamos definição do modelo/funções e execução do treino + validação.

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

input_dim = X_train.shape[1]
num_classes = 2

train_params = {
    'epochs': 15,
    'batch_size': 32,
    'learning_rate': 1e-1,
}

mlp_configs = {
    'Config A': {
        'hidden_dims': [2],
        'dropout': 0.1,
        'weight_decay': 0.0,
    },
    'Config B': {
        'hidden_dims': [16, 8],
        'dropout': 0.2,
        'weight_decay': 1e-3,
    }
}

print(f"Input dim: {input_dim} | Classes: {num_classes}")
print("Parâmetros de treino:")
display(pd.DataFrame([train_params]))

print("Configurações comparadas:")
display(pd.DataFrame(mlp_configs).T)

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.0, num_classes=2):
        super().__init__()
        layers = []
        prev = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=train_params['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=train_params['batch_size'], shuffle=False)


def train_mlp_model(_, config):
    model = MLPModel(
        input_dim=input_dim,
        hidden_dims=config['hidden_dims'],
        dropout=config['dropout'],
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=train_params['learning_rate'],
        weight_decay=config['weight_decay']
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    best_val_loss = float('inf')
    best_state = None

    for _ in range(train_params['epochs']):
        model.train()
        train_loss_sum, train_correct, train_total = 0.0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * yb.size(0)
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)

                val_loss_sum += loss.item() * yb.size(0)
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == yb).sum().item()
                val_total += yb.size(0)

        epoch_train_loss = train_loss_sum / train_total
        epoch_val_loss = val_loss_sum / val_total
        epoch_train_acc = train_correct / train_total
        epoch_val_acc = val_correct / val_total

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history, best_val_loss

In [ ]:
mlp_runs = {}
for cfg_name, cfg in mlp_configs.items():
    model, history, best_val = train_mlp_model(cfg_name, cfg)
    mlp_runs[cfg_name] = {
        'model': model,
        'history': history,
        'best_val_loss': best_val,
    }

resumo_validacao = pd.DataFrame([
    {'config': name, 'best_val_loss': run['best_val_loss']}
    for name, run in mlp_runs.items()
]).sort_values('best_val_loss').reset_index(drop=True)

mlp_best_name = resumo_validacao.loc[0, 'config']
mlp_best_model = mlp_runs[mlp_best_name]['model']

display(resumo_validacao)
print(f"Melhor configuração na validação: {mlp_best_name}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cfg_name, run in mlp_runs.items():
    axes[0].plot(run['history']['train_loss'], label=f"{cfg_name} - train")
    axes[0].plot(run['history']['val_loss'], linestyle='--', label=f"{cfg_name} - val")

axes[0].set_title('Loss por época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)

for cfg_name, run in mlp_runs.items():
    axes[1].plot(run['history']['train_acc'], label=f"{cfg_name} - train")
    axes[1].plot(run['history']['val_acc'], linestyle='--', label=f"{cfg_name} - val")

axes[1].set_title('Acurácia por época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 1.4 Métricas
*Acurácia, Precisão, Recall, F1 e Matriz de Confusão.*


In [ ]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_np = y_test.values

results = []
conf_mats = {}

for cfg_name, run in mlp_runs.items():
    model = run['model']
    model.eval()

    with torch.no_grad():
        logits = model(X_test_tensor)
        y_pred = torch.argmax(logits, dim=1).cpu().numpy()

    acc = accuracy_score(y_test_np, y_pred)
    prec = precision_score(y_test_np, y_pred, average='binary', zero_division=0)
    rec = recall_score(y_test_np, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_test_np, y_pred, average='binary', zero_division=0)
    cm = confusion_matrix(y_test_np, y_pred)

    results.append({
        'config': cfg_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
    })
    conf_mats[cfg_name] = cm

results_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
results_df_fmt = results_df.copy()
for col in ['accuracy', 'precision', 'recall', 'f1']:
    results_df_fmt[col] = results_df_fmt[col].map(lambda x: f"{x:.4f}")

print('Resumo de métricas no teste:')
display(results_df_fmt)

print('Matrizes de confusão (tabelas):')
for cfg_name, cm in conf_mats.items():
    print(f"\n{cfg_name}")
    display(pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1']))

### 1.5 Análise

Pelos resultados, as duas configurações tiveram desempenho aceitável, com vantagem da Configuração B. Na validação, ela ficou com menor perda (`best_val_loss = 0.030271` contra `0.043761` da Config A), e no teste também terminou melhor (`accuracy = 0.9884` e `f1 = 0.9841`, contra `0.9767` e `0.9688`). Nenhuma configuração obteve acúracia >= 99%, mas o resultado geral se mostra aceitável.

Também é importante considerar o tamanho do dataset. A entrada é pequena e, depois da divisão, a validação fica menor ainda. Nesse contexto, com um ajuste razoável de arquitetura e regularização, o modelo tende a errar pouco porque há menos casos e o padrão do problema é mais direto. Além disso, a redução de features provavelmente ajudou a diminuir ruído e facilitou a convergência da rede.

Uma MLP é uma escolha válida nesse problema porque o dado é tabular, numérico e não tem dados faltantes (não validei, mas é dito na documentação do [dataset](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)). Não há estrutura espacial de imagem nem relação de grafos para justificar outro tipo de arquitetura (CNN, GAN e/ou GNN).

Por fim, em problema de diagnóstico não devemos olhar apenas para a acurácia. Neste caso, não há uma distribuição extremamente desigual, a validação inicial foi feita justamente pensando nesse tipo de risco, que é uma possibilidade real em datasets de dados tabulares. Dependendo da distribuição das classes, é possível ter acurácia alta e ainda assim falhar em casos críticos. Por isso, além da acurácia, a avaliação com precisão, recall, F1 e matriz de confusão foi necessária para enxergar melhor o risco real dos erros.

---
## Questão 2: CNN - Fashion-MNIST


### 2.1 Descrição e entendimento do problema

O dataset do [Fashion-MNIST](https://www.kaggle.com/datasets/zalando-research/fashionmnist/data) reúne imagens (matrizes de pixels) de roupas, com um target (a classe da peça de roupa). Novamente, faremos uma análise inicial do dataset (aliado a exploração de sua documentação) para entender melhor o contexto do problema antes de partirmos para a modelagem da rede convolucional. 


In [ ]:
train_raw = FashionMNIST(root='./data', train=True, download=True)
test_raw = FashionMNIST(root='./data', train=False, download=True)

print(f'Tipo do dataset: {type(train_raw)}')
print(f'Tamanho treino: {len(train_raw)} | Tamanho teste: {len(test_raw)}\n')

label_names = train_raw.classes
print(label_names)
print(f'Mapeamento classe->id: {train_raw.class_to_idx}\n')

x0 = train_raw.data[0].unsqueeze(0).float() / 255.0
y0 = int(train_raw.targets[0].item())
C, H, W = x0.shape
print(f'Shape de uma amostra x: {tuple(x0.shape)} -> C={C}, H={H}, W={W}')
print(f'Dtype de x: {x0.dtype}')
print(f'Faixa de x: [{x0.min().item():.3f}, {x0.max().item():.3f}]\n')
print('Fica claro que os dados, de fato, estao normalizados para [0,1] nessa etapa')

train_counts = pd.Series(train_raw.targets.numpy()).value_counts().sort_index()
test_counts = pd.Series(test_raw.targets.numpy()).value_counts().sort_index()
dist_df = pd.DataFrame({
    'target_id': range(len(label_names)),
    'target_str': label_names,
    'train_count': train_counts.values,
    'test_count': test_counts.values,
})
dist_df['train_pct'] = (dist_df['train_count'] / len(train_raw) * 100).round(2)
dist_df['test_pct'] = (dist_df['test_count'] / len(test_raw) * 100).round(2)

display(dist_df)
print(f"Soma treino: {dist_df['train_count'].sum()} | Soma teste: {dist_df['test_count'].sum()}")

X_flat = train_raw.data.view(len(train_raw), -1).numpy().astype(np.float32) / 255.0
pixel_cols = [f'px_{i}' for i in range(784)]
df_tab = pd.DataFrame(X_flat, columns=pixel_cols)
df_tab['target_id'] = train_raw.targets.numpy()

print(f'Shape: {df_tab.shape} - (784 pixels + target_id)')
display(pd.concat([df_tab.dtypes.head(2), df_tab.dtypes.tail(2)]))

display(df_tab.head(3))

df_vis = df_tab.copy()
df_vis['target_str'] = df_vis['target_id'].map(lambda i: label_names[i])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls in range(len(label_names)):
    idx = np.where(train_raw.targets.numpy() == cls)[0][0]
    img = (train_raw.data[idx].float() / 255.0).numpy()
    lbl = int(train_raw.targets[idx].item())
    axes[cls // 5, cls % 5].imshow(img, cmap='gray')
    axes[cls // 5, cls % 5].set_title(f'{lbl} - {label_names[lbl]}')
    axes[cls // 5, cls % 5].axis('off')
plt.tight_layout()
plt.show()

Na exploração inicial, a estrutura da base foi confirmada por etapas: tipo do objeto, tamanho de treino/teste, mapeamento das classes e formato real da entrada. Com a conversão para float e divisão por 255, cada imagem fica como `(C,H,W) = (1,28,28)`, com pixel em `[0,1]`, e target numérico de 0 a 9.

Também foi montada uma visão tabular para inspeção dos dados: 784 colunas de pixel e 1 coluna de target (`target_id`), totalizando 785 colunas. O nome da classe (`target_str`) foi mantido apenas como apoio de visualização, sem entrar no dataset principal.

Outro ponto importante é o tamanho da base. Em comparação com a Questão 1 (MLP, base tabular médica menor), aqui há 60.000 amostras de treino e 10.000 de teste. Além disso, a distribuição por classe veio equilibrada, sem sinal de desbalanceamento nessa etapa inicial.

### 2.2 Preparação dos Dados

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

X_all = train_raw.data.unsqueeze(1).float() / 255.0
y_all = train_raw.targets.long()

X_all = (X_all - 0.5) / 0.5
X_test = test_raw.data.unsqueeze(1).float() / 255.0
X_test = (X_test - 0.5) / 0.5
y_test_fashion = test_raw.targets.long()

train_idx, val_idx = train_test_split(
    np.arange(len(X_all)),
    test_size=0.15,
    random_state=seed,
    stratify=y_all.numpy()
)

full_train = TensorDataset(X_all, y_all)
train_dataset = torch.utils.data.Subset(full_train, train_idx)
val_dataset = torch.utils.data.Subset(full_train, val_idx)
test_dataset = TensorDataset(X_test, y_test_fashion)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

sample_x, sample_y = next(iter(train_loader))
print(f'Shape batch treino: {tuple(sample_x.shape)}')
print(f'Shape labels batch treino: {tuple(sample_y.shape)}')
print(f'Tamanho treino: {len(train_dataset)} | validacao: {len(val_dataset)} | teste: {len(test_dataset)}')
print('Preparacao concluida.')

### 2.3 Configuração + Treinamento + Validação

Nesta etapa ficam juntas a definição das configurações, a arquitetura da CNN e o loop de treino/validação. A proposta é comparar duas configurações sob o mesmo protocolo (mesma divisão e mesmos hiperparâmetros-base), acompanhar as curvas por época e selecionar o melhor estado pelo menor valor de loss na validação.

A escolha por uma arquitetura CNN em vez de uma MLP para este problema é justificada pela natureza espacial e estrutural das imagens do dataset em questão. Enquanto a MLP exige o achatamento de uma imagem em um vetor unidimensional, o que é problemático para a relação de vizinhança entre os pixels e causa uma explosão no número de parâmetros, a CNN preserva a matriz bidimensional do dado. Utilizando conectividade local e compartilhamento de pesos através do kernel, a CNN é capaz de capturar características, como bordas, costuras e etc, de forma eficiente, além de mitigar invariância à mudança na posição do artefato em análise, o que permite o modelo reconhecer a peça mesmo que ela sofra deslocamentos.

In [ ]:
num_classes = 10
input_channels = 1

cnn_train_params = {
    'epochs': 8,
    'learning_rate': 1e-3,
    'batch_size': batch_size,
}

cnn_configs = {
    'Config A': {
        'channels': [16, 32],
        'fc_dim': 64,
        'dropout': 0.1,
        'weight_decay': 0.0,
    },
    'Config B': {
        'channels': [32, 64, 128],
        'fc_dim': 128,
        'dropout': 0.3,
        'weight_decay': 1e-4,
    }
}

class CNNModel(nn.Module):
    def __init__(self, channels, fc_dim, dropout=0.0, num_classes=10):
        super().__init__()

        conv_blocks = []
        in_ch = 1
        for out_ch in channels:
            conv_blocks.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1))
            conv_blocks.append(nn.ReLU())
            conv_blocks.append(nn.MaxPool2d(2))
            in_ch = out_ch
        self.features = nn.Sequential(*conv_blocks)

        self.gap = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels[-1] * 4 * 4, fc_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_dim, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

print('Configuracoes da CNN:')
display(pd.DataFrame(cnn_configs).T)
print('Parametros de treino CNN:')
display(pd.DataFrame([cnn_train_params]))

In [ ]:
def evaluate_loader(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            preds = torch.argmax(logits, dim=1)
            loss_sum += loss.item() * yb.size(0)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
            all_preds.append(preds.cpu())
            all_targets.append(yb.cpu())
    avg_loss = loss_sum / total
    acc = correct / total
    return avg_loss, acc, torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy()

def train_cnn_model(config):
    model = CNNModel(
        config['channels'],
        fc_dim=config['fc_dim'],
        dropout=config['dropout'],
        num_classes=num_classes
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=cnn_train_params['learning_rate'],
        weight_decay=config['weight_decay']
    )

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None

    for _ in range(cnn_train_params['epochs']):
        model.train()
        train_loss_sum, train_correct, train_total = 0.0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            preds = torch.argmax(logits, dim=1)
            train_loss_sum += loss.item() * yb.size(0)
            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total
        val_loss, val_acc, _, _ = evaluate_loader(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history, best_val_loss

cnn_runs = {}
for cfg_name, cfg in cnn_configs.items():
    model, history, best_val = train_cnn_model(cfg)
    cnn_runs[cfg_name] = {
        'model': model,
        'history': history,
        'best_val_loss': best_val,
    }

cnn_val_df = pd.DataFrame([
    {'config': name, 'best_val_loss': run['best_val_loss']}
    for name, run in cnn_runs.items()
]).sort_values('best_val_loss').reset_index(drop=True)

cnn_best_name = cnn_val_df.loc[0, 'config']
cnn_best_model = cnn_runs[cnn_best_name]['model']

display(cnn_val_df)
print(f'Melhor configuracao na validacao: {cnn_best_name}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cfg_name, run in cnn_runs.items():
    axes[0].plot(run['history']['train_loss'], label=f'{cfg_name} - train')
    axes[0].plot(run['history']['val_loss'], linestyle='--', label=f'{cfg_name} - val')
axes[0].set_title('Loss por epoca')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)

for cfg_name, run in cnn_runs.items():
    axes[1].plot(run['history']['train_acc'], label=f'{cfg_name} - train')
    axes[1].plot(run['history']['val_acc'], linestyle='--', label=f'{cfg_name} - val')
axes[1].set_title('Acuracia por epoca')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Acuracia')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 2.4 Métricas

A avaliação considera acurácia, matriz de confusão e exemplos qualitativos de acertos/erros do melhor modelo. Essa combinação evita leitura limitada ao valor final de acurácia e permite observar com mais clareza o padrão dos erros.

In [ ]:
cnn_results = []
cnn_conf_mats = {}

for cfg_name, run in cnn_runs.items():
    model = run['model']
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, y_pred, y_true = evaluate_loader(model, test_loader, criterion)
    cm = confusion_matrix(y_true, y_pred)

    cnn_results.append({
        'config': cfg_name,
        'test_loss': test_loss,
        'accuracy': test_acc,
    })
    cnn_conf_mats[cfg_name] = cm

cnn_results_df = pd.DataFrame(cnn_results).sort_values('accuracy', ascending=False).reset_index(drop=True)
cnn_results_fmt = cnn_results_df.copy()
for c in ['test_loss', 'accuracy']:
    cnn_results_fmt[c] = cnn_results_fmt[c].map(lambda x: f'{x:.4f}')

print('Resumo no teste:')
display(cnn_results_fmt)

print('Matrizes de confusao:')
for cfg_name, cm in cnn_conf_mats.items():
    print(f'\n{cfg_name}')
    display(pd.DataFrame(cm, index=[f'Real {i}' for i in range(10)], columns=[f'Pred {i}' for i in range(10)]))

best_cfg = cnn_best_name
best_cm = cnn_conf_mats[best_cfg]

off_diag = []
for real_id in range(best_cm.shape[0]):
    for pred_id in range(best_cm.shape[1]):
        if real_id != pred_id and best_cm[real_id, pred_id] > 0:
            off_diag.append({
                'real_id': real_id,
                'pred_id': pred_id,
                'real': label_names[real_id],
                'pred': label_names[pred_id],
                'erros': int(best_cm[real_id, pred_id]),
            })

top_confusions_df = pd.DataFrame(off_diag).sort_values('erros', ascending=False).head(10).reset_index(drop=True)
print(f'\nTop confusoes do melhor modelo ({best_cfg}):')
display(top_confusions_df)

best_model = cnn_runs[best_cfg]['model']
best_model.eval()

x_test_all, y_test_all = [], []
for xb, yb in test_loader:
    x_test_all.append(xb)
    y_test_all.append(yb)
x_test_all = torch.cat(x_test_all)
y_test_all = torch.cat(y_test_all)

with torch.no_grad():
    logits = best_model(x_test_all.to(device)).cpu()
    probs = torch.softmax(logits, dim=1)
    pred_all = torch.argmax(probs, dim=1)
    conf_all = probs.max(dim=1).values

correct_idx = (pred_all == y_test_all).nonzero(as_tuple=True)[0]
wrong_idx = (pred_all != y_test_all).nonzero(as_tuple=True)[0]

n_show = 6

def pick_diverse_correct(indices, true_labels, scores, max_items):
    selected = []
    used_classes = set()
    ranked = indices[torch.argsort(scores[indices], descending=True)]

    for idx in ranked.tolist():
        cls = int(true_labels[idx].item())
        if cls not in used_classes:
            selected.append(idx)
            used_classes.add(cls)
        if len(selected) == max_items:
            break

    if len(selected) < max_items:
        for idx in ranked.tolist():
            if idx not in selected:
                selected.append(idx)
            if len(selected) == max_items:
                break

    return np.array(selected, dtype=int)

def pick_diverse_wrong(indices, true_labels, pred_labels, scores, max_items):
    selected = []
    used_pairs = set()
    ranked = indices[torch.argsort(scores[indices], descending=True)]

    for idx in ranked.tolist():
        pair = (int(true_labels[idx].item()), int(pred_labels[idx].item()))
        if pair not in used_pairs:
            selected.append(idx)
            used_pairs.add(pair)
        if len(selected) == max_items:
            break

    if len(selected) < max_items:
        for idx in ranked.tolist():
            if idx not in selected:
                selected.append(idx)
            if len(selected) == max_items:
                break

    return np.array(selected, dtype=int)

sel_correct = pick_diverse_correct(correct_idx, y_test_all, conf_all, n_show) if len(correct_idx) > 0 else np.array([], dtype=int)
sel_wrong = pick_diverse_wrong(wrong_idx, y_test_all, pred_all, conf_all, n_show) if len(wrong_idx) > 0 else np.array([], dtype=int)

def denorm_for_plot(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

fig, axes = plt.subplots(2, n_show, figsize=(3.5 * n_show, 7.2), constrained_layout=True)
fig.suptitle(f'Melhor modelo: {best_cfg} | Acertos e erros representativos', fontsize=13)

for i in range(n_show):
    axes[0, i].axis('off')
    axes[1, i].axis('off')

for i, idx in enumerate(sel_correct):
    img = denorm_for_plot(x_test_all[idx]).squeeze(0)
    t = y_test_all[idx].item()
    p = pred_all[idx].item()
    conf = conf_all[idx].item()
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'Acerto ({conf:.2f})\nR:{label_names[t]} | P:{label_names[p]}', fontsize=8)
    axes[0, i].axis('off')

for i, idx in enumerate(sel_wrong):
    img = denorm_for_plot(x_test_all[idx]).squeeze(0)
    t = y_test_all[idx].item()
    p = pred_all[idx].item()
    conf = conf_all[idx].item()
    prob_true = probs[idx, t].item()
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(
        f'Erro ({conf:.2f}) p(real)={prob_true:.2f}\nR:{label_names[t]} | P:{label_names[p]}',
        fontsize=8
    )
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Acertos', fontsize=11)
axes[1, 0].set_ylabel('Erros', fontsize=11)
plt.show()

### 2.5 Análise

A comparação foi conduzida de forma consistente: as duas configurações passaram pelo mesmo pipeline e a seleção do melhor modelo foi feita pela validação, não pelo teste. Nesse cenário, a Config B ficou à frente da Config A tanto nas curvas quanto no resultado final, indicando ganho real de capacidade de representação, e não vantagem ocasional.

Pelas curvas de treino e validação, não apareceu sinal forte de underfitting, porque ambas aprendem de forma estável ao longo das épocas. Também não houve evidência de overfitting severo no recorte adotado, já que a validação acompanha a tendência do treino sem abrir um gap persistente e explosivo. Ainda assim, a diferença entre treino e validação existe e merece monitoramento em execuções mais longas, especialmente na Config B.

A matriz de confusão e a tabela de maiores erros mostram um padrão claro: os enganos se concentram em classes visualmente próximas, principalmente entre peças de roupa superior. Nesta execução, os pares mais críticos foram T-shirt/top -> Shirt (104), Shirt -> T-shirt/top (89), Shirt -> Coat (87), Pullover -> Coat (69), Shirt -> Pullover (69) e Coat -> Shirt (53). Esse comportamento é coerente com o Fashion-MNIST, que trabalha com imagens pequenas (28x28), em tons de cinza, com variação de pose, recorte e contraste.

Do ponto de vista de arquitetura, a CNN se encaixa melhor neste problema porque explora estrutura espacial da imagem: convoluções aprendem padrões locais úteis para discriminar classes, pooling reduz sensibilidade a pequenas variações e compartilhamento de pesos melhora eficiência com menos parâmetros do que uma rede totalmente conectada na entrada. A principal limitação permanece nos casos de fronteira visual muito parecida; por isso, mesmo com bom desempenho global, a análise por classe continua indispensável para validar a qualidade real do modelo.

---
## Questão 3: GAN - MNIST


### 3.1 Descrição e entendimento do problema


Nem sempre o objetivo de redes neurais é predição, em muitos casos queremos gerar novos dados que se pareçam com dados reais já existentes e possam ser usados em tarefas futuras de classificação, análise ou enriquecimento de datasets. Nesse cenário, GANs são um instrumento poderoso, pois permitem aprender a distribuição dos dados reais e gerar novas amostras sinteticamente que mantêm características semelhantes (a partir de certo ponto no treinamento).

Uma GAN funciona a partir de redes adversárias, sendo elas o Gerador, que cria imagens falsas a partir de vetores do vetor latente, e o Discriminador, que tenta distinguir imagens reais (do dataset) das imagens geradas. Nesse processo competitivo, ambas melhoram iterativamente até que o Gerador consegue produzir amostras capazes de enganar o Discriminador.

Comparando duas configurações de GAN, uma mais simples e outra com maior capacidade, poderemos observar como a arquitetura influencia a qualidade das amostras geradas.

O [dataset](https://www.kaggle.com/datasets/hojjatk/mnist-dataset) em questão é um bom ponto de partida pois é uma base clássica de redes neurais, com imagens simples e 10 dígitos manuscritos. Percebemos que as imagens tem uma geometria e formas simplificadas, o que permite uma análise e evolução do processo de treinamento das redes, especialmente a rede Geradora, que tem a tarefa de aprender a criar dígitos que se pareçam com os reais.

### 3.2 Implementação e Configuração


In [ ]:
mnist_raw = MNIST(root='./data', train=True, download=True)

X_mnist = mnist_raw.data.unsqueeze(1).float() / 255.0
X_mnist = (X_mnist - 0.5) / 0.5
y_mnist = mnist_raw.targets.long() 

batch_size_gan = 128
gan_dataset = TensorDataset(X_mnist, y_mnist)
gan_loader = DataLoader(gan_dataset, batch_size=batch_size_gan, shuffle=True)

class DiscriminatorA(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x): 
        return self.net(x)

class GeneratorA(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 28*28),
            nn.Tanh()
        )
    def forward(self, x):
        return self.net(x).view(-1, 1, 28, 28)

class DiscriminatorB(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x): 
        return self.net(x)

class GeneratorB(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 28*28),
            nn.Tanh()
        )
    def forward(self, x):
        return self.net(x).view(-1, 1, 28, 28)

gan_configs = {
    'Config A': {
        'z_dim': 64,
        'lr': 0.0002,
        'beta1': 0.9, 
        'epochs': 30,
        'D_class': DiscriminatorA,
        'G_class': GeneratorA
    },
    'Config B': {
        'z_dim': 100,
        'lr': 0.0002,
        'beta1': 0.2,
        'epochs': 30,
        'D_class': DiscriminatorB,
        'G_class': GeneratorB
    }
}
print(f"Base carregada. Formato X: {X_mnist.shape}, Formato y: {y_mnist.shape}")
print("Configurações das arquiteturas A e B definidas.")

### 3.3 Treino e Amostras Obtidas

In [ ]:
criterion_gan = nn.BCELoss()

def train_gan_model(nome_config, config, loader):
    print(f"Treinando {nome_config}...")
    
    D = config['D_class']().to(device)
    G = config['G_class'](config['z_dim']).to(device)
    
    opt_D = optim.Adam(D.parameters(), lr=config['lr'], betas=(config['beta1'], 0.999))
    opt_G = optim.Adam(G.parameters(), lr=config['lr'], betas=(config['beta1'], 0.999))
    
    epochs = config['epochs']
    z_dim = config['z_dim']
    
    fixed_z = torch.randn(16, z_dim).to(device)
    
    hist = {'d_loss': [], 'g_loss': []}
    amostras_checkpoints = {}
    
    pontos_extracao = [0, epochs // 2, epochs - 1]
    
    for ep in range(epochs):
        D.train()
        G.train()
        
        d_loss_ep = 0.0
        g_loss_ep = 0.0
        
        for real_imgs, _ in loader:
            real_imgs = real_imgs.to(device)
            bs = real_imgs.size(0)
            
            label_real = torch.ones(bs, 1).to(device)
            label_fake = torch.zeros(bs, 1).to(device)
            
            opt_D.zero_grad()
            
            out_real = D(real_imgs)
            loss_d_real = criterion_gan(out_real, label_real)
            
            z = torch.randn(bs, z_dim).to(device)
            fake_imgs = G(z)
            out_fake = D(fake_imgs.detach())
            loss_d_fake = criterion_gan(out_fake, label_fake)
            
            loss_d = loss_d_real + loss_d_fake
            loss_d.backward()
            opt_D.step()
            
            opt_G.zero_grad()
            
            out_fake_for_g = D(fake_imgs)
            loss_g = criterion_gan(out_fake_for_g, label_real)
            
            loss_g.backward()
            opt_G.step()
            
            d_loss_ep += loss_d.item()
            g_loss_ep += loss_g.item()
            
        hist['d_loss'].append(d_loss_ep / len(loader))
        hist['g_loss'].append(g_loss_ep / len(loader))
        
        if ep in pontos_extracao:
            G.eval()
            with torch.no_grad():
                imgs_gen = ((G(fixed_z).cpu() * 0.5) + 0.5).clamp(0, 1)
                
                if ep == 0:
                    estagio = 'Início'
                elif ep == epochs // 2:
                    estagio = 'Estágio Intermediário'
                else:
                    estagio = 'Etapa Final'
                    
                amostras_checkpoints[estagio] = imgs_gen
                
    return hist, amostras_checkpoints

gan_resultados = {}
for nome, cfg in gan_configs.items():
    hist, amostras = train_gan_model(nome, cfg, gan_loader)
    gan_resultados[nome] = {'hist': hist, 'amostras': amostras}

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle("Evolução das Amostras (Vetor Latente Fixo)", fontsize=16)

estagios = list(list(gan_resultados.values())[0]['amostras'].keys())

for idx_estagio, estagio in enumerate(estagios):
    for idx_config, (nome_cfg, res) in enumerate(gan_resultados.items()):
        
        imgs = res['amostras'][estagio]
        grid = make_grid(imgs, nrow=4, padding=2).numpy()
        grid = np.transpose(grid, (1, 2, 0))
        
        axes[idx_estagio, idx_config].imshow(grid, cmap='gray')
        axes[idx_estagio, idx_config].set_title(f"{nome_cfg}\n({estagio})")
        axes[idx_estagio, idx_config].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Curvas de Custo Adversário (Loss)", fontsize=14)

for idx_cfg, (nome_cfg, res) in enumerate(gan_resultados.items()):
    ax = axes[idx_cfg]
    ax.plot(res['hist']['g_loss'], label='Loss G (Gerador)', color='royalblue')
    ax.plot(res['hist']['d_loss'], label='Loss D (Discriminador)', color='darkorange')
    ax.set_title(nome_cfg)
    ax.set_xlabel("Épocas")
    ax.set_ylabel("Binary Cross Entropy")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

real_batch, _ = next(iter(gan_loader))
real_batch_viz = ((real_batch.cpu() * 0.5) + 0.5).clamp(0, 1)[:9]  

fig, axes = plt.subplots(1, 3, figsize=(14, 8))

grid_real = make_grid(real_batch_viz, nrow=3, padding=2).numpy()
grid_real = np.transpose(grid_real, (1, 2, 0)).squeeze()
axes[0].imshow(grid_real, cmap='gray')
axes[0].set_title("Imagens Reais", fontsize=11, fontweight='bold')
axes[0].axis('off')

imgs_a_final = gan_resultados['Config A']['amostras']['Etapa Final'][:9]
grid_a = make_grid(imgs_a_final, nrow=3, padding=2).numpy()
grid_a = np.transpose(grid_a, (1, 2, 0)).squeeze()
axes[1].imshow(grid_a, cmap='gray')
axes[1].set_title("Config A ", fontsize=11, fontweight='bold')
axes[1].axis('off')

imgs_b_final = gan_resultados['Config B']['amostras']['Etapa Final'][:9]
grid_b = make_grid(imgs_b_final, nrow=3, padding=2).numpy()
grid_b = np.transpose(grid_b, (1, 2, 0)).squeeze()
axes[2].imshow(grid_b, cmap='gray')
axes[2].set_title("Config B", fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 3.4 Análise

A diferença entre as duas configurações é nítida, embora ambas ainda fiquem longe da perfeição do dataset real. A Configuração A entregou amostras ruidosas e sem estrutura definida. Usar dados ruins assim para aumento de dados é perigoso, pois cai no clássico garbage in, garbage out, inserindo lixo na base e prejudicando o desempenho de um classificador posterior. A Configuração B foi muito melhor e gerou dígitos legíveis na etapa final, mas ainda exibe falhas consideráveis, como nos números `x` e `y`, que ficaram com a estrutura visivelmente prejudicada.

Esse comportamento se reflete direto nas curvas de perda. Na Configuração A, a loss do Gerador sofre oscilações com picos, gerando saltos bruscos e desconexos a cada estágio visual a partir do ruído inicial. Na Configuração B, a loss do gerador cai e satura rápido. Essa estabilização precoce explica por que as imagens mudam pouco entre o estágio intermediário e a etapa final. Isso mostra que o aprendizado estagnou e que simplesmente aumentar o número de épocas não adiantaria. Assim, o caminho para corrigir esses números deformados seria de fato aprofundar a rede.

As escolhas de design justificam esses resultados. A Configuração A usou ReLU simples com beta1=0.9, tornando o otimizador muito inercial e travado. A Configuração B combinou LeakyReLU(0.2), que deixa os gradientes passarem mesmo no lado negativo e evita que a rede morra, BatchNorm para estabilizar as ativações internas do gerador, e um beta1 menor (0.2) para dar mais agilidade ao algoritmo contra as mudanças do adversário. O dropout no discriminador também foi vital para que ele não memorizasse o dataset real e esmagasse o gerador antes da hora.

Fica evidente que as GANs são ferramentas poderosas para modelagem generativa e síntese de dados, mas o sucesso depende de combinar a capacidade e a profundidade do modelo com a complexidade do padrão que se tenta replicar (dataset real observado).

---
## Questão 4: GNN - Cora

### 4.1 Descrição e entendimento do problema

O [dataset](https://relational.fit.cvut.cz/dataset/CORA) em questão representa uma rede de citações de artigos científicos. Diferente dos problemas anteriores onde cada dado era isolado em uma tabela (Questão 1) ou numa grade de pixels independente (Questões 2 e 3), aqui temos um problema relacional, onde a informação de um artigo ajuda a classificar outro.


- **Nós:** representam os artigos científicos individuais documentados na rede.
- **Arestas:** representam os relacionamentos diretos de citação (seja quando o artigo cita alguém ou quando é citado de volta).
- **Alvo:** cada artigo deve ser categorizado corretamente em uma dentre as 7 áreas de computação oferecidas (Machine Learning, Theory, etc).

Esse problema é naturalmente relacional porque as áreas de estudo afetam quem cita quem na prática. 

In [ ]:
dataset = Planetoid(root='./data/Cora', name='Cora')
data = dataset[0]

print(f"Número de artigos (Nós): {data.num_nodes}")
print(f"Número de citações (Arestas): {data.num_edges}")
print(f"Número de atributos por artigo (Features): {dataset.num_node_features}")
print(f"Áreas possíveis (Classes): {dataset.num_classes}\n")

print(f"A base é direcionada? {data.is_directed()}")
print(f"Tem nós isolados? {data.has_isolated_nodes()}")

### 4.2 Preparação dos Dados

No PyTorch Geometric (e processamento de grafos no geral) nós separamos os dados principais em estruturas básicas:
- **$X$:** É a matriz contendo o atributo de cada nó individual. No caso do Cora, o valor de $X$ é composto por indicações de palavras-chave (um vocabulário fixo convertido em variáveis flag). Isso confere a estrutura ao nó isolado.
- **$A$:** Essa é a topologia do grafo. O PyTorch lida com isso salvando uma lista bruta com os pares de conexões que existirem, batizada de `edge_index` (no formato nó de origem e nó de destino), o que torna isso viável sem ter que processar uma matriz de adjacência $N \times N$, poupando memória ao armazenarmos um recorte com vários milhões de nós para lidar com buracos estáticos enormes (vazios da matriz onde conexões não se ligam).
- **Embeddings de nós:** Representações vetoriais otimizadas que o motor da GNN mapeia durante o loop misturando o que há na feature originária do nó ($X$) com os perfis filtrados a partir das restrições e influências das ligações topológicas dele ($A$). 

Enquanto MLPs lidam batendo arrays cortados numa rotina e uma divisão estrita cega (sem conexões), num grafo fica impossível rasgar uma imagem que une arestas em todos os caminhos; eles tem de ficar todos conectados de uma única vez alimentando nossa rede inteiramente para haver difusão plena na camada conectada e não causar furos estelares durante propagação. Todo mundo se processa em bloco conjunto da entrada ($X$ com base em seu respectivo $A$), os limites de treino, test e valid acontecem utilizando matrizes binárias chamadas **Máscaras**, delimitando como a métrica do erro é filtrada a cada parte processual e o resultado validado no final.

In [ ]:
X_cora = data.x
A_cora = data.edge_index
Y_cora = data.y

print(f"Dimensão matriz de atributos X: {tuple(X_cora.shape)}")
print(f"Dimensão lista de conexões A: {tuple(A_cora.shape)}")
print(f"Dimensão do target Y: {tuple(Y_cora.shape)}\n")

print(f"Máscara de Treino: {(data.train_mask).sum().item()} nós classificados ativamente durante treino")
print(f"Máscara de Validação: {(data.val_mask).sum().item()} nós usados na perda de validacao")
print(f"Máscara de Teste: {(data.test_mask).sum().item()} nós isolados pro resultado de teste")

data = data.to(device)

### 4.3 Configuração e Treinamento

A arquitetura escolhida é a Graph Convolutional Network (GCN). Como queríamos analisar fisicamente o comportamento do número de camadas no efeito final comparatório, avaliaremos:
- **Configuração A:** Uma GCN Padrão contendo 2 camadas de convolução de grafos (geralmente ótima métrica inicial).
- **Configuração B:** Uma GCN "Extensa", totalizando 4 camadas GCNConv. A ideia é criar a oportunidade estrutural de flagrar ativamente algum enfraquecimento relacional por excesso no fluxo processado pela rede, também muito rotulado por *over-smoothing* onde a densidade se sobrepuja até as classes colapsarem similarmente sem relevância local de atributos próprios.

Treinaremos com **Cross Entropy Loss** e otimizador **Adam**. A dimensão dos embeddings intermediários foi fixada em **16** e utilizamos **ReLU** como função de ativação entre as camadas. Mantivemos esses hiperparâmetros padronizados nas duas configurações para isolarmos e medirmos estritamente o impacto causado pela variação no número de camadas.

In [ ]:
class GCNModel(nn.Module):
    def __init__(self, in_features, hidden_dim, out_classes, num_layers, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.dropout = dropout
        
        self.convs.append(GCNConv(in_features, hidden_dim))
        
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            
        self.convs.append(GCNConv(hidden_dim, out_classes))

    def forward(self, x, edge_index):
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            
        x = self.convs[-1](x, edge_index)
        return x

cora_configs = {
    'Config A (2 camadas)': {'num_layers': 2, 'hidden_dim': 16, 'epochs': 200, 'lr': 0.01, 'weight_decay': 5e-4},
    'Config B (4 camadas)': {'num_layers': 4, 'hidden_dim': 16, 'epochs': 200, 'lr': 0.01, 'weight_decay': 5e-4}
}

criterion_gcn = nn.CrossEntropyLoss()

def train_gcn_model(config, data):
    model = GCNModel(
        in_features=dataset.num_node_features,
        hidden_dim=config['hidden_dim'],
        out_classes=dataset.num_classes,
        num_layers=config['num_layers']
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_state = None
    
    for epoch in range(config['epochs']):
        model.train()
        optimizer.zero_grad()
        
        out = model(data.x, data.edge_index)
        
        loss_train = criterion_gcn(out[data.train_mask], data.y[data.train_mask])
        loss_train.backward()
        optimizer.step()
        
        preds = out.argmax(dim=1)
        acc_train = (preds[data.train_mask] == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()
        
        model.eval()
        with torch.no_grad():
            out_val = model(data.x, data.edge_index)
            loss_val = criterion_gcn(out_val[data.val_mask], data.y[data.val_mask])
            acc_val = (preds[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()
            
        hist['train_loss'].append(loss_train.item())
        hist['val_loss'].append(loss_val.item())
        hist['train_acc'].append(acc_train)
        hist['val_acc'].append(acc_val)
        
        if loss_val.item() < best_val_loss:
            best_val_loss = loss_val.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            
    model.load_state_dict(best_state)
    return model, hist, best_val_loss

gcn_runs = {}
for name, cfg in cora_configs.items():
    model, hist, best_val = train_gcn_model(cfg, data)
    gcn_runs[name] = {'model': model, 'hist': hist, 'best_val': best_val}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cfg_name, res in gcn_runs.items():
    axes[0].plot(res['hist']['train_loss'], label=f"{cfg_name} (Train)")
    axes[0].plot(res['hist']['val_loss'], linestyle='--', label=f"{cfg_name} (Val)")
axes[0].set_title("Loss por Época")
axes[0].set_xlabel("Épocas")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)

for cfg_name, res in gcn_runs.items():
    axes[1].plot(res['hist']['train_acc'], label=f"{cfg_name} (Train)")
    axes[1].plot(res['hist']['val_acc'], linestyle='--', label=f"{cfg_name} (Val)")
axes[1].set_title("Acurácia por Época")
axes[1].set_xlabel("Épocas")
axes[1].set_ylabel("Acurácia")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 4.4 Métricas

Nessa bateria avaliaremos a acurácia no extrato de nós exclusivos marcados pela lógica binária da matriz teste `test_mask`, além de extrairmos o relatório analítico (matriz de confusão) oriundo do modelo ideal parametrizado internamente e finalizado.

In [ ]:
gcn_results = []
conf_mats_gcn = {}

for name, res in gcn_runs.items():
    model = res['model']
    model.eval()
    
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        preds = out.argmax(dim=1)
        
        test_mask = data.test_mask
        acc = (preds[test_mask] == data.y[test_mask]).sum().item() / test_mask.sum().item()
        
        y_true = data.y[test_mask].cpu().numpy()
        y_pred = preds[test_mask].cpu().numpy()
        
        conf_mat = confusion_matrix(y_true, y_pred)
        
        gcn_results.append({'Config': name, 'Acurácia Teste': f"{acc:.4f}"})
        conf_mats_gcn[name] = conf_mat

print("Resumo final no teste isolado de nós:")
display(pd.DataFrame(gcn_results).set_index('Config'))

print("\nMatriz de Confusão (Config A):")
display(pd.DataFrame(conf_mats_gcn['Config A (2 camadas)'], 
                     index=[f'Real {i}' for i in range(dataset.num_classes)], 
                     columns=[f'Pred {i}' for i in range(dataset.num_classes)]))

### 4.5 Análise

A topologia do grafo é essencial aqui porque as informações individuais (o abstract ou título do artigo, extraídos em palavras-chave $X$) recebem uma "validação de vizinhança". Quando iteramos uma camada, é exatamente isso que significa "propagação de informação entre vizinhos": o nó engloba as características ponderadas dos nós que conectam-se ativamente com ele de acordo com $A$. Consequentemente, se os vizinhos (quem o artigo cita) pertencem a Machine Learning, naturalmente a rede deduz que o artigo corrente caminha na mesma direção. É o ditado "diga-me com quem andas", mas executado através do kernel de cálculo estritamente matemático contra pesos aprendidos. Ignorar esse fator em uma MLP mataria as features conectivas e dependeríamos religiosamente em apenas encontrar palavras sozinhas no atributo matriz; na GNN o viés das arestas eleva monstruosamente a precisão conectando os pontos.

As escolhas configuradas entregam uma leitura visual valiosíssima sobre esse tipo computacional de estrutura e propagação. Pelos resultados capturados nos gráficos e reportados na matriz de confusão final, a Configuração A brilhou em seu minimalismo e conquistou 78.7% de acurácia. A evolução das curvas mostra um aprendizado sadio, onde a rede alcançou o ápice logo cedo sem gerar underfitting. Em um dataset coeso como o Cora, 2 camadas de convolução (GCN) acabam atuando perfeitamente por limitarem a expansão da mensagem em até dois nós de distância. É um raio ótimo, limitando que a agregação pare antes do limite tênue de ruídos estranhos e focando exatamente onde há coesão relacional relevante de citações diretas.

Em contraponto, o que rola em redes de imagem onde mais camadas quase equivalem a melhores resultados, não é realidade em vizinhança gráfica estática. A Configuração B com suas 4 camadas puxou a acurácia para 76.10% em resposta à fadiga por profundidade. Como a base natural do Cora possui atalhos muito fechados de citação interligando toda a extensão teórica, dar 4 recortes expansivos causa uma salada de features que afeta severamente o grafo. Isso é classificado matematicamente como *over-smoothing*, onde tudo fica tão espremido contra seus vizinhos que a própria estrutura da informação relacional e das próprias palavras-chave sofre um desgaste genérico. Todo mundo termina virando parte idêntica do mesmo "blob" estatístico indistinguível e o poder de discriminação de nicho é apagado, prejudicando o teste.

---
## Questão 5 - Arquiteturas em Ciência de Dados



## a)

#### MLP
* **Tipo de dado mais adequado:** Dados tabulares estruturados (matrizes de linhas e colunas) onde as variáveis representam as features (as quais geralmente são independentes), sem correlação de proximidade espacial ou dependência temporal direta.
* **Tarefa principal:** Predição supervisionada: classificação de classes e regressão de valores contínuos.
* **Principal força:** Atua como um aproximador de funções para problemas de baixa e média complexidade. Sua implementação é direta e serve como um baseline antes da adoção de modelos especializados.
* **Principal limitação:** A conectividade total entre suas camadas causa aumento significativo no volume de parâmetros quando a dimensão da entrada cresce. Além disso, é sensível à escolha de hiperparâmetros e vulnerável ao *overfitting* (ou *underfitting*) se o ajuste inicial não for preciso.

#### CNN
* **Tipo de dado mais adequado:** Dados estruturados em formato bi ou tridimensional que apresentam uma relação de vizinhança e dependência espacial entre seus elementos, como imagens, vídeos e espectrogramas.
* **Tarefa principal:** Extração automática de características e reconhecimento de padrões visuais (bordas, texturas e formas geométricas).
* **Principal força:** Preserva a topologia bidimensional original do dado sem destruir a relação geométrica dos elementos. O uso de filtros convolucionais com compartilhamento de pesos reduz de forma drástica o número de parâmetros necessários em comparação com uma MLP, além de conferir invariância à translação.
* **Principal limitação:** É projetada para topologias de grade regular, sendo assim ineficiente para dados tabulares comuns. Também não lida nativamente com dependências temporais ou sequenciais de longo prazo, o que historicamente cabe a RNN ou aos Transformers.

#### GAN
* **Tipo de dado mais adequado:** Amostras de alta dimensionalidade (imagens, áudios ou textos) cuja distribuição estatística original se deseja aprender e mapear para a criação de instâncias artificiais inéditas.
* **Tarefa principal:** Modelagem generativa implícita, focada na síntese e geração de novos dados sintéticos a partir de vetores de ruído em um espaço latente.
* **Principal força:** Capacidade única de criar novos registros realistas e plausíveis que simulam o comportamento estatístico da base original. É uma ferramenta estratégica para o enriquecimento de datasets e simulação de cenários.
* **Principal limitação:** Possui uma dinâmica de treinamento complexa e instável baseada em um jogo de soma zero entre o Gerador e o Discriminador. O tempo de processamento é significativamente superior ao de redes preditivas convencionais, exigindo minutos de processamento onde outras redes operam em segundos, e o modelo é vulnerável a falhas como o *mode collapse*.

#### GNN
* **Tipo de dado mais adequado:** Dados estruturados como grafos, nós e arestas, onde a informação reside tanto nos atributos individuais de cada entidade quanto nas suas conexões e relacionamentos.
* **Tarefa principal:** Aprendizado de representações relacionais e mapeamento de redes, atuando na classificação de nós, predição de conexões e classificação.
* **Principal força:** Habilidade de fundir atributos locais com a estrutura global do grafo por meio do mecanismo de message passing, lidando com problemas de interdependência.
* **Principal limitação:** Elevado custo computacional para escalar o algoritmo em grafos massivos ou dinâmicos. Além disso, a rede sofre com *oversmoothing* caso camadas demais sejam adicionadas, o que torna as representações dos nós estatisticamente idênticas e destrói o poder preditivo do modelo.

---

### Resumo 

| Arquitetura | Dado Ideal | Tarefa Principal | Principal Força | Principal Limitação |
| :--- | :--- | :--- | :--- | :--- |
| **MLP** | Tabelas estruturadas (sem relação espacial/temporal). | Classificação e regressão direta de alvos preditivos. | Simplicidade e atuação como aproximador universal. | Explosão de parâmetros em alta dimensão e sensibilidade a hiperparâmetros. |
| **CNN** | Grades 2D/3D com correlação de pixels (Imagens/Vídeos). | Extração de características e reconhecimento visual. | Compartilhamento de pesos e preservação da vizinhança. | Inadequada para tabelas e ausência de tracking temporal nativo. |
| **GAN** | Amostras complexas para replicação estatística. | Geração e síntese de dados realistas e artificiais. | Criação de dados sintéticos plausíveis para enriquecimento (*augmentation*). | Treinamento muito lento e instável, sujeito a colapso de modo (*mode collapse*). |
| **GNN** | Estruturas de grafos e redes (Nós e Conexões relacionais). | Classificação de entidades e predição de links complexos. | Fusão nativa de atributos de nós com a topologia local (*message passing*). | Dificuldade de escala em grafos gigantescos e perda de distinção por *oversmoothing*. |

### b)

#### MLP vs. CNN em Dados Visuais
Em dados visuais, a CNN apresenta uma vantagem esmagadora sobre a MLP devido à sua capacidade de processar a dependência espacial nativa dos pixels. Pixels geometricamente próximos possuem forte correlação semântica, e os filtros da CNN são desenhados justamente para extrair esses padrões locais de forma hierárquica. 

Por outro lado, a MLP exige o achatamento (*flatten*) da matriz da imagem em um vetor unidimensional, destruindo completamente a relação de vizinhança física entre os elementos. Além disso, a conectividade total da MLP faz com que o número de parâmetros exploda conforme a resolução da imagem cresce, levando o modelo à perda de generalização e ao *overfitting* (memorização da base). A CNN mitiga isso com o compartilhamento de pesos, tornando o aprendizado eficiente. 

*Nota sobre dados dinâmicos:* Vale destacar que o domínio visual não se limita a imagens estáticas. No caso de vídeos, além da dependência espacial de cada frame (tratada pela CNN), há uma forte dependência temporal entre os frames ao longo do tempo. Para capturar essa dinâmica sequencial de forma ideal, a abordagem clássica exige associar a CNN a uma Rede Neural Recorrente (RNN/LSTM) ou a mecanismos de atenção.

#### CNN vs. GAN em Problemas com Imagens
A comparação entre CNN e GAN no processamento de imagens não estabelece uma relação de superioridade, mas sim de divergência absoluta de objetivos. Ambas operam sobre a mesma estrutura de dados (grades de pixels), porém em sentidos matemáticos opostos.

A CNN é um modelo discriminativo: sua tarefa principal é receber uma imagem complexa de alta dimensionalidade e reduzi-la a um rótulo ou probabilidade de classe (predição e classificação). A GAN é uma arquitetura generativa: ela faz o caminho inverso, partindo de um vetor de ruído simples (espaço latente) para construir uma imagem inédita e complexa que simule o comportamento estatístico do dataset real. Embora o estado da arte atual utilize modelos mais recentes (como Vision Transformers para classificação e Modelos de Difusão para geração), a divisão conceitual permanece: escolhe-se a CNN para extrair informações e tomar decisões baseadas em imagens, e a GAN para sintetizar novos dados visuais.

#### MLP vs. GNN em Dados Relacionais
Quando lidamos com dados relacionais, onde o valor da informação está na interconexão entre as entidades, a GNN supera a MLP por ter sido projetada especificamente para topologias não-euclidianas. Um exemplo clássico são as redes de citações acadêmicas, onde o fato de um artigo apontar para outro cria uma relação estrutural explícita que define o contexto do dado.

A MLP falha nesse cenário porque opera sob a premissa de que as linhas de uma tabela são instâncias independentes e isoladas. Ela processa os atributos de um registro, mas é incapaz de enxergar com quem aquele registro se relaciona no ecossistema. A GNN resolve esse gargalo ao utilizar o mecanismo de passagem de mensagens (*message passing*), que funde as características de um nó com a estrutura de sua vizinhança. Assim, para dados estruturados em redes ou grafos, a GNN consegue mapear a topologia das conexões de forma que uma MLP jamais conseguiria fazer nativamente.

### c)

A seleção de uma arquitetura de Aprendizado Profundo não deve se pautar pelo apelo midiático de modelos de estado da arte (SOTA) ou pela complexidade estrutural bruta de uma rede. No ecossistema de Ciência de Dados, o processo de modelagem é um problema de otimização sujeito a restrições multifatoriais. O engenheiro ou cientista de dados deve avaliar de forma síncrona a topologia do dado, o mapeamento matemático do objetivo, o nível exigido de interpretabilidade, a eficiência de FLOPs (custo computacional) e as barreiras teóricas do modelo. A premissa de que maior complexidade técnica implica melhores soluções é empiricamente falsa.

#### I. Alinhamento Topológico do Dado e a Natureza do Objetivo
Cada arquitetura neural possui um viés indutivo (*inductive bias*) nativo, projetado para explorar propriedades geométricas ou estruturais específicas dos dados. Forçar uma rede totalmente conectada (MLP) a processar dados com dependências espaciais (como imagens) ou sequenciais (como séries temporais ou vídeos) viola esse princípio. O código executará e os gradientes serão computados, mas a rede será forçada a aprender invariâncias estruturais fundamentais a partir do zero, resultando em uma convergência ineficiente, superfícies de perda caóticas e alta propensão ao *overfitting*. 

Ademais, a natureza do dado não dita unicamente a escolha do modelo; o objetivo matemático da tarefa é o fator decisivo. Tomando como exemplo o dataset MNIST: se o problema consiste em um mapeamento discriminativo tradicional ($f: X \to Y$) para prever classes discretas, uma CNN estruturada é a escolha correta pela sua capacidade de extração hierárquica de características locais. Contudo, se o escopo muda para a modelagem da distribuição de probabilidade implícita do dataset a fim de sintetizar novas instâncias ($P(X)$), o problema exige uma abordagem generativa, como o equilíbrio adversarial de uma GAN. O dado de entrada é estritamente o mesmo, mas a divergência entre os objetivos analítico e sintético demanda formulações matemáticas e fluxos de gradientes totalmente opostos.

#### II. O Paradoxo da Capacidade vs. Interpretabilidade
A interpretabilidade é uma métrica de validação crítica em ambientes de produção, sobretudo em domínios de alta criticidade ou altamente regulados, como o setor financeiro e a medicina diagnóstica. Arquiteturas profundas operam fundamentalmente como modelos de caixa-preta (*black boxes*). À medida que adicionamos camadas ocultas e funções de ativação não-lineares, o espaço de características é projetado em dimensões abstratas de altíssima ordem, tornando quase impossível rastrear a atribuição de importância das variáveis (*feature attribution*) ou extrair uma justificativa causal explícita para uma inferência específica.

Se analisarmos o cenário abordado na Questão 1, a classificação de tumores (benignos ou malignos) com base em 23 variáveis clínicas, a aplicação de uma MLP complexa ou de arquiteturas mais densas sacrifica a transparência em troca de uma capacidade matemática desnecessária. Em cenários médicos, compreender o peso relativo de cada parâmetro clínico na decisão do modelo é um requisito tão vital quanto a acurácia bruta da predição. Modelos preditivos mais simples ou com menor profundidade oferecem superfícies de decisão mais inteligíveis e permitem extrair correlações diretas, mitigando o risco de decisões baseadas em vieses ocultos nas camadas latentes.

#### III. Eficiência Computacional e o Custo de Inferência em Produção
O custo computacional estabelece o limite de viabilidade econômica e operacional de qualquer projeto de Ciência de Dados. Como verificado experimentalmente, enquanto arquiteturas preditivas direcionadas convergiram em poucos segundos, o treinamento de uma GAN demandou ordens de magnitude a mais de tempo e processamento devido à otimização simultânea e concorrente de duas redes neurais distintas (Gerador e Discriminador) jogando um jogo de soma zero.

Essa discrepância de desempenho torna-se um gargalo crítico quando escalada para ambientes de produção industrial, impactando diretamente o provisionamento de hardware (GPUs/TPUs), a latência de inferência em tempo real e o consumo energético. Seria tecnicamente viável empregar um modelo massivo baseado em *Vision Transformers* (ViTs) ou arquiteturas autoregressivas complexas para realizar a classificação de dígitos no MNIST. No entanto, do ponto de vista da engenharia, isso representa um desperdício flagrante de recursos. Uma CNN compacta, desenhada sob medida com camadas convolucionais e pooling adequados, atinge o mesmo limite teórico de acurácia utilizando uma fração ínfima do custo computacional e de memória. A excelência em engenharia de dados consiste em arquitetar a solução de menor complexidade que satisfaça as restrições do problema com máxima eficiência.